In [ ]:
import os, sys, subprocess
print(sys.version)
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU chưa bật. Vào Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# Clone repo (use actual path - user will replace owner/repo)
REPO_URL = "https://github.com/<owner>/<repo>.git"
BRANCH = "main"
!git clone -b {BRANCH} {REPO_URL}
%cd husc-admission-chat-enrollment/rag2025
!pip install -r requirements.txt -q

In [ ]:
import sys
sys.path.insert(0, "/content/husc-admission-chat-enrollment")
# Set PYTHONPATH
os.environ["PYTHONPATH"] = "/content/husc-admission-chat-enrollment"
# Smoke import
from rag2025.src.main import UnifiedQueryRequest, UnifiedQueryResponse
print("Import OK")

In [ ]:
# Env vars from Colab secrets/runtime
# User sets these in Colab "Secrets" tab or runtime
import os
BASE_URL = os.getenv("PIPELINE_BASE_URL", "http://127.0.0.1:8000")
PRIMARY = "results/test_questions.json"
FALLBACK = "backup_mail_package_2026/python_project/rag2025/results/test_questions.json"
# API keys (user fills in Colab secrets)
API_KEY = os.getenv("RAMCLOUDS_API_KEY", "") or os.getenv("OPENAI_API_KEY", "")

In [ ]:
from notebooks.eval_core import load_test_questions
rows, used_path = load_test_questions(PRIMARY, FALLBACK)
print(f"Loaded {len(rows)} questions from {used_path}")
# Show category breakdown
from collections import Counter
cats = Counter(r["category"] for r in rows)
print(f"Category breakdown: {dict(cats)}")

In [ ]:
from notebooks.eval_core import call_pipeline, normalize_pipeline_output, should_abort_after_smoke
smoke_results = []
n_smoke = 5
for i, item in enumerate(rows[:n_smoke]):
    try:
        raw = call_pipeline(BASE_URL, item["question"], mode="v2", top_k=5)
        norm = normalize_pipeline_output(raw, mode="v2")
        smoke_results.append({"id": item.get("id", i), "question": item["question"], "answer": norm["answer"], "ok": bool(norm["answer"])})    except Exception as e:
        smoke_results.append({"id": item.get("id", i), "question": item["question"], "answer": "", "ok": False, "error": str(e)})

failures = sum(1 for r in smoke_results if not r["ok"])
print(f"Smoke: {n_smoke} questions, {failures} failures")
if should_abort_after_smoke(n_smoke, failures):
    raise RuntimeError(f"Smoke failed: >50% failure rate ({failures}/{n_smoke}). Check pipeline connection.")
print("Smoke OK")

In [ ]:
full_results = []
for i, item in enumerate(rows):
    try:
        raw = call_pipeline(BASE_URL, item["question"], mode="v2", top_k=5)
        norm = normalize_pipeline_output(raw, mode="v2")
        full_results.append({**item, **norm})
    except Exception as e:
        full_results.append({**item, "answer": "", "error": str(e)})
    if (i+1) % 10 == 0:
        print(f"Progress: {i+1}/{len(rows)}")

print(f"Full eval complete: {len(full_results)} results")

In [ ]:
from notebooks.eval_core import exact_correctness, retrieval_recall_proxy, hallucination_flag, GROUNDING_THRESHOLD
import pandas as pd
import json
import os

os.makedirs("results/colab_eval", exist_ok=True)

# Score each result
scored = []
for r in full_results:
    gt = r.get("ground_truth_answer", "")
    pred = r.get("answer", "")
    source_ids = r.get("source_ids", [])
    gt_chunks = r.get("ground_truth_source_chunks", [])
    groundedness = r.get("groundedness_score", 0.0)
    
    score_exact = exact_correctness(pred, gt)
    recall = retrieval_recall_proxy(source_ids, gt_chunks)
    halluc = hallucination_flag(groundedness, GROUNDING_THRESHOLD)
    
    scored.append({**r, "score_exact": score_exact, "recall": recall, "hallucination": halluc})

# Save eval_scored.jsonl
with open("results/colab_eval/eval_scored.jsonl", "w", encoding="utf-8") as f:
    for r in scored:
        f.write(json.dumps(r, ensure_ascii=False) + "
")

# Save eval_predictions.jsonl (minimal)
with open("results/colab_eval/eval_predictions.jsonl", "w", encoding="utf-8") as f:
    for r in scored:
        f.write(json.dumps({"id": r.get("id"), "question": r["question"], "answer": r.get("answer",""), "mode": "v2"}, ensure_ascii=False) + "
")

# Summary CSV
summary = []
for cat in ["simple", "multihop"]:
    subset = [r for r in scored if r.get("category") == cat]
    if subset:
        summary.append({"category": cat, "count": len(subset), "accuracy": sum(r["score_exact"] for r in subset)/len(subset), "recall": sum(r["recall"] for r in subset)/len(subset), "hallucination_rate": sum(r["hallucination"] for r in subset)/len(subset)})

pd.DataFrame(summary).to_csv("results/colab_eval/eval_summary_metrics.csv", index=False)
print("Exported: eval_predictions.jsonl, eval_scored.jsonl, eval_summary_metrics.csv")
print(pd.DataFrame(summary))

In [ ]:
from notebooks.eval_core import build_diagnostic_report
import json

with open("results/colab_eval/eval_scored.jsonl", "r", encoding="utf-8") as f:
    scored_data = [json.loads(line) for line in f]

result = {
    "summary": {"accuracy": sum(r["score_exact"] for r in scored_data)/len(scored_data) if scored_data else 0},
    "errors": [r for r in scored_data if r["score_exact"] == 0][:5]
}
report = build_diagnostic_report(result)
with open("results/colab_eval/diagnostic_report.md", "w", encoding="utf-8") as f:
    f.write(report)
print("Report saved: diagnostic_report.md")
print(report[:500])